# 🇨🇱 Análisis del PIB de Chile (1961–2024)

Exploración visual del crecimiento económico de Chile comparado con el promedio mundial y Corea del Sur.  
Los datos provienen del **Banco Mundial** vía Kaggle, con estimaciones propias para 2021–2024.

**Contenido:**
1. Configuración e importaciones
2. Carga y procesamiento de datos
3. Gráfica 1 — Crecimiento acumulado: Chile vs Mundo
4. Gráfica 2 — Crecimiento anual con períodos presidenciales
5. Gráfica 3 — Chile vs Mundo vs Corea del Sur
6. Gráfica 4 — Top 9 economías del mundo (2020)

---
**Fuente de datos:** [GDP of all countries (1960–2020)](https://www.kaggle.com/datasets/zgrcemta/world-gdpgdp-gdp-per-capita-and-annual-growths) · World Bank via Kaggle

## 1. Configuración e importaciones

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Alta resolución para visualizar en el notebook
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── Paleta de colores ────────────────────────────────────────────────────────
CHILE_COLOR     = '#3EBCD2'
WORLD_COLOR     = '#006BA2'
KOREA_COLOR     = '#EBB434'
ACCENT_COLOR    = '#E3120B'
PRESIDENT_COLOR = '#A81829'
GRID_COLOR      = '#758D99'

# ── Datos recientes (post-dataset) ───────────────────────────────────────────
RECENT_CHILE = [(2021, 11.7), (2022, 2.4), (2023, -0.53), (2024, 0.7)]
RECENT_WORLD = [(2021, 6.0),  (2022, 3.2), (2023, 3.0),   (2024, 3.1)]

# ── Lista de presidentes con año de inicio ───────────────────────────────────
PRESIDENTS = [
    (1960, 'Alessandri'), (1964, 'Frei M.'),  (1970, 'Allende'),
    (1974, 'Pinochet'),   (1990, 'Aylwin'),   (1994, 'Frei R.'),
    (2000, 'Lagos'),      (2006, 'Bachelet'), (2010, 'Piñera'),
    (2014, 'Bachelet'),   (2018, 'Piñera'),   (2022, 'Boric'),
]

SOURCE = 'Fuente: World Bank — "GDP of all countries (1960–2020)" vía Kaggle.com | 2021–2024 estimados'

print("Librerías cargadas correctamente.")

## 2. Carga y procesamiento de datos

El dataset principal (`gdp_growth.csv`) tiene una fila por país y una columna por año.  
Lo transponemos para obtener series de tiempo limpias.

In [ ]:
def extract_country(gdp, code, recent=None):
    """
    Extrae la serie anual de crecimiento del PIB para un país dado.
    - gdp   : DataFrame wide (países × años)
    - code  : código ISO-3 del país (ej. 'CHL', 'KOR')
    - recent: lista de tuplas (año, valor) para extender la serie
    """
    df = gdp[gdp['Code'] == code].T.iloc[2:].reset_index()
    df.columns = ['año', 'gdp']
    df = df.iloc[:-1].dropna()
    df['año'] = df['año'].astype(float)
    df['gdp'] = df['gdp'].astype(float)
    if recent:
        df = pd.concat(
            [df, pd.DataFrame(recent, columns=['año', 'gdp'])],
            ignore_index=True,
        )
    return df


def extract_world(gdp, recent=None):
    """Calcula el promedio mundial de crecimiento del PIB por año."""
    world = gdp.loc[:, '1961':'2020'].mean().reset_index()
    world.columns = ['año', 'gdp']
    world['año'] = world['año'].astype(float)
    if recent:
        world = pd.concat(
            [world, pd.DataFrame(recent, columns=['año', 'gdp'])],
            ignore_index=True,
        )
    return world


# Cargar dataset
gdp_growth = pd.read_csv('data/gdp_growth.csv')

# Extraer series
chile = extract_country(gdp_growth, 'CHL', RECENT_CHILE)
world = extract_world(gdp_growth, RECENT_WORLD)
korea = extract_country(gdp_growth, 'KOR')

print(f"Chile: {len(chile)} años  ({int(chile['año'].min())}–{int(chile['año'].max())})")
print(f"Mundo: {len(world)} años  ({int(world['año'].min())}–{int(world['año'].max())})")
print(f"Corea: {len(korea)} años  ({int(korea['año'].min())}–{int(korea['año'].max())})")
chile.tail(5)

### Funciones de estilo reutilizables

Centralizamos el estilo visual (inspirado en *The Economist*) para mantener consistencia entre todas las gráficas.

In [ ]:
def style_ax(ax):
    """Aplica el estilo base a un eje: grid, spines y tick labels."""
    ax.grid(which='major', axis='y', color=GRID_COLOR, alpha=0.35, zorder=1,
            linestyle='--', linewidth=0.7)
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(axis='both', labelsize=10, color=GRID_COLOR)


def add_header(ax, fig, title, subtitle, y_line=1.16, y_title=1.08, y_sub=1.03):
    """Línea roja de acento + título + subtítulo al estilo The Economist."""
    ax.plot([0.12, 0.90], [y_line, y_line],
            transform=fig.transFigure, clip_on=False,
            color=ACCENT_COLOR, linewidth=0.6)
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.12, y_line - 0.025), 0.055, 0.025,
        boxstyle='square,pad=0', facecolor=ACCENT_COLOR,
        transform=fig.transFigure, clip_on=False, linewidth=0,
    ))
    ax.text(0.12, y_title, title, transform=fig.transFigure,
            ha='left', fontsize=13, weight='bold', alpha=0.88)
    ax.text(0.12, y_sub, subtitle, transform=fig.transFigure,
            ha='left', fontsize=10.5, alpha=0.72)


def add_footer(ax, fig, source=SOURCE, y=0.01):
    ax.text(0.12, y, source, transform=fig.transFigure,
            ha='left', fontsize=8.5, alpha=0.58)


def add_presidential_markers(ax, ymin, fontsize=8.5):
    """Líneas verticales punteadas con el nombre de cada presidente."""
    for year, name in PRESIDENTS:
        ax.axvline(x=year, color=PRESIDENT_COLOR, linestyle=':',
                   linewidth=0.9, alpha=0.65, zorder=2)
        ax.text(year + 0.3, ymin, name, rotation=90, va='bottom',
                fontsize=fontsize, color=PRESIDENT_COLOR, alpha=0.82)


def add_covid_band(ax):
    """Banda gris semitransparente para el período COVID-19."""
    ax.axvspan(2019.5, 2021.5, color='#9BB7D4', alpha=0.20, linewidth=0, zorder=1)


print("Funciones de estilo definidas.")

## 3. Gráfica 1 — Crecimiento acumulado: Chile vs Mundo

Sumamos los porcentajes anuales para ver cómo se acumula el crecimiento a lo largo del tiempo.  
Un valor acumulado más alto indica que ese país o grupo creció más en total desde 1961.

In [ ]:
c = chile.copy()
w = world.copy()
c['gdp_sum'] = c['gdp'].cumsum()
w['gdp_sum'] = w['gdp'].cumsum()

fig, ax = plt.subplots(figsize=(10, 5))
style_ax(ax)

ax.plot(c['año'], c['gdp_sum'], color=CHILE_COLOR, linewidth=2.5, zorder=3)
ax.plot(w['año'], w['gdp_sum'], color=WORLD_COLOR, linewidth=2.5, zorder=3)
ax.axhline(0, color='#333333', linewidth=0.6, alpha=0.45)

# Etiquetas al final de cada línea
for df, color, label in [(c, CHILE_COLOR, 'Chile'), (w, WORLD_COLOR, 'Mundial')]:
    last = df.iloc[-1]
    ax.annotate(
        f"{label}  {last['gdp_sum']:.0f}%",
        xy=(last['año'], last['gdp_sum']),
        xytext=(6, 0), textcoords='offset points',
        color=color, fontsize=10, va='center', fontweight='bold',
    )

ax.set_ylabel('Crecimiento acumulado (%)', fontsize=10, alpha=0.75)
add_header(ax, fig,
           title='PIB de Chile',
           subtitle='Crecimiento acumulado comparado con el promedio mundial (1961–2024)')
add_footer(ax, fig)

plt.savefig('images/chile_growth.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Gráfica 2 — Crecimiento anual % con períodos presidenciales

Visualizamos la volatilidad año a año del PIB chileno y lo contextualizamos con:
- **Líneas punteadas rojas**: cambio de gobierno
- **Banda gris**: período COVID-19 (2020–2021)
- **Área verde/roja**: diferencia entre crecimiento positivo y contracción

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
style_ax(ax)

# Áreas positivas/negativas
ax.fill_between(chile['año'], chile['gdp'], 0,
                where=(chile['gdp'] >= 0),
                alpha=0.12, color=CHILE_COLOR, zorder=2)
ax.fill_between(chile['año'], chile['gdp'], 0,
                where=(chile['gdp'] < 0),
                alpha=0.20, color=ACCENT_COLOR, zorder=2)

add_covid_band(ax)

ax.plot(chile['año'], chile['gdp'], color=CHILE_COLOR, linewidth=2.5, zorder=3, label='Chile')
ax.plot(world['año'], world['gdp'], color=WORLD_COLOR, linewidth=1.8,
        linestyle='--', zorder=3, label='Promedio mundial')
ax.axhline(0, color='#333333', linewidth=0.7, alpha=0.45)

ymin = chile['gdp'].min() - 4
add_presidential_markers(ax, ymin=ymin)

ax.legend(frameon=False, fontsize=10, loc='upper right',
          handles=[
              mpatches.Patch(color=CHILE_COLOR, label='Chile'),
              mpatches.Patch(color=WORLD_COLOR, label='Promedio mundial'),
              mpatches.Patch(color='#9BB7D4', alpha=0.6, label='COVID-19'),
          ])
ax.set_ylabel('Crecimiento anual (%)', fontsize=10, alpha=0.75)
add_header(ax, fig,
           title='El modelo chileno',
           subtitle='Crecimiento anual del PIB (%) por período presidencial (1961–2024)')
add_footer(ax, fig)

plt.savefig('images/chile_growth_index.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Gráfica 3 — Chile vs Mundo vs Corea del Sur

Corea del Sur es un caso de estudio frecuente en economía del desarrollo: partió desde niveles similares a Chile en los años 60 y alcanzó crecimientos extraordinarios.  
Esta comparación ilustra trayectorias distintas con puntos de partida parecidos.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
style_ax(ax)

add_covid_band(ax)

ax.plot(chile['año'], chile['gdp'], color=CHILE_COLOR, linewidth=2.5, zorder=3, label='Chile')
ax.plot(world['año'], world['gdp'], color=WORLD_COLOR, linewidth=1.8,
        linestyle='--', zorder=3, label='Promedio mundial')
ax.plot(korea['año'], korea['gdp'], color=KOREA_COLOR, linewidth=2,
        linestyle='-.', zorder=3, label='Corea del Sur')
ax.axhline(0, color='#333333', linewidth=0.7, alpha=0.45)

ymin = min(chile['gdp'].min(), korea['gdp'].min()) - 4
add_presidential_markers(ax, ymin=ymin)

ax.legend(frameon=False, fontsize=10, loc='upper right',
          handles=[
              mpatches.Patch(color=CHILE_COLOR, label='Chile'),
              mpatches.Patch(color=WORLD_COLOR, label='Promedio mundial'),
              mpatches.Patch(color=KOREA_COLOR, label='Corea del Sur'),
              mpatches.Patch(color='#9BB7D4', alpha=0.6, label='COVID-19'),
          ])
ax.set_ylabel('Crecimiento anual (%)', fontsize=10, alpha=0.75)
add_header(ax, fig,
           title='Chile vs Corea del Sur',
           subtitle='Comparación del crecimiento anual del PIB (%) con períodos presidenciales (1961–2024)')
add_footer(ax, fig)

plt.savefig('images/chile_growth_index_KR.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Gráfica 4 — Top 9 economías del mundo (2020)

Gráfico de barras horizontales al estilo *The Economist* con las 9 economías de mayor PIB en 2020.  
Se muestran las etiquetas de valor en USD billones para facilitar la lectura.

In [ ]:
gdp_abs = pd.read_csv('data/gdp_1960_2020.csv')
gdp_abs['gdp_trillions'] = gdp_abs['gdp'] / 1e12
gdp_abs['country'] = gdp_abs['country'].replace('the United States', 'United States')
gdp_bar = gdp_abs[gdp_abs['year'] == 2020].sort_values('gdp_trillions').tail(9).copy()

fig, ax = plt.subplots(figsize=(4.5, 6))
ax.grid(which='major', axis='x', color=GRID_COLOR, alpha=0.35, zorder=1,
        linestyle='--', linewidth=0.7)
ax.spines[['top', 'right', 'bottom']].set_visible(False)
ax.spines['left'].set_linewidth(1.1)

bars = ax.barh(gdp_bar['country'], gdp_bar['gdp_trillions'],
               color=WORLD_COLOR, zorder=2, height=0.6)

# Etiquetas de valor en cada barra
for bar, val in zip(bars, gdp_bar['gdp_trillions']):
    ax.text(bar.get_width() + 0.25,
            bar.get_y() + bar.get_height() / 2,
            f'${val:.1f}T',
            va='center', fontsize=9.5, color='#333333')

ax.set_xticks([0, 5, 10, 15, 20])
ax.xaxis.set_tick_params(labeltop=True, labelbottom=False,
                         bottom=False, labelsize=10, pad=-1)
ax.set_yticklabels(gdp_bar['country'], ha='left')
ax.yaxis.set_tick_params(pad=110, labelsize=10, bottom=False)
ax.set_ylim(-0.5, 8.5)
ax.set_xlim(0, 25)

add_header(ax, fig,
           title='Las grandes ligas',
           subtitle='PIB 2020, billones de USD',
           y_line=1.09, y_title=1.01, y_sub=0.965)
add_footer(ax, fig, y=0.04)

plt.savefig('images/economist_bar.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()